In [ ]:
# Step 1: Import required library
import numpy as np
import random

# Step 2: Define Tic-Tac-Toe environment
class TicTacToe:

    def __init__(self):
        self.reset()

    # Initialize an empty board
    def reset(self):
        self.board = [' '] * 9
        return self.get_state()

    # Convert board into a string state
    def get_state(self):
        return ''.join(self.board)

    # Check whether a player has won
    def check_winner(self, player):
        win_positions = [
            (0, 1, 2), (3, 4, 5), (6, 7, 8),
            (0, 3, 6), (1, 4, 7), (2, 5, 8),
            (0, 4, 8), (2, 4, 6)
        ]

        for a, b, c in win_positions:
            if (self.board[a] == player and
                self.board[b] == player and
                self.board[c] == player):
                return True

        return False

    # Check whether the game is over
    def is_done(self):
        return (
            self.check_winner('X') or
            self.check_winner('O') or
            ' ' not in self.board
        )

    # Perform a move
    def make_move(self, action, player):
        if self.board[action] == ' ':
            self.board[action] = player
            return True

        return False

    # Display the board
    def display(self):
        for i in range(0, 9, 3):
            print(' | '.join(self.board[i:i+3]))
            if i < 6:
                print('--+---+--')


# Step 3: Define Q-Learning Agent
class QLearningAgent:

    def __init__(self):
        self.q_table = {}

        self.alpha = 0.1       # Learning rate
        self.gamma = 0.9       # Discount factor
        self.epsilon = 1.0     # Exploration rate

    # Get Q-values for a state
    def get_q_values(self, state):
        if state not in self.q_table:
            self.q_table[state] = np.zeros(9)

        return self.q_table[state]

    # Select action using epsilon-greedy policy
    def choose_action(self, state, available_actions):
        if random.random() < self.epsilon:
            return random.choice(available_actions)

        q_values = self.get_q_values(state)

        best_value = max(q_values[a] for a in available_actions)

        best_actions = [
            a for a in available_actions
            if q_values[a] == best_value
        ]

        return random.choice(best_actions)

    # Update Q-value using Q-Learning formula
    def learn(self, state, action, reward, next_state, done):

        current_q = self.get_q_values(state)[action]

        if done:
            max_next_q = 0
        else:
            max_next_q = np.max(self.get_q_values(next_state))

        new_q = current_q + self.alpha * (
            reward + self.gamma * max_next_q - current_q
        )

        self.q_table[state][action] = new_q


# Step 4: Train the agent
env = TicTacToe()
agent = QLearningAgent()

episodes = 50000

for episode in range(episodes):

    state = env.reset()
    done = False

    while not done:

        # AI player X selects a move
        available = [
            i for i in range(9) if env.board[i] == ' '
        ]

        action = agent.choose_action(state, available)
        env.make_move(action, 'X')

        # Check if AI wins
        if env.check_winner('X'):
            agent.learn(state, action, 1, env.get_state(), True)
            break

        # Check for draw
        if ' ' not in env.board:
            agent.learn(state, action, 0, env.get_state(), True)
            break

        next_state = env.get_state()

        # Random opponent O selects a move
        opponent_action = random.choice([
            i for i in range(9) if env.board[i] == ' '
        ])

        env.make_move(opponent_action, 'O')

        # Check if opponent wins
        if env.check_winner('O'):
            agent.learn(state, action, -1, env.get_state(), True)
            break

        # Check for draw
        if ' ' not in env.board:
            agent.learn(state, action, 0, env.get_state(), True)
            break

        # Update Q-table
        next_state = env.get_state()

        agent.learn(
            state, action, 0,
            next_state, False
        )

        state = next_state

    # Reduce exploration gradually
    agent.epsilon = max(0.05, agent.epsilon * 0.9999)


# Step 5: Test the trained agent
agent.epsilon = 0

wins = 0
losses = 0
draws = 0

test_games = 100

for game in range(test_games):

    env.reset()
    done = False

    while not done:

        # AI player X
        available = [
            i for i in range(9) if env.board[i] == ' '
        ]

        state = env.get_state()
        action = agent.choose_action(state, available)

        env.make_move(action, 'X')

        if env.check_winner('X'):
            wins += 1
            break

        if ' ' not in env.board:
            draws += 1
            break

        # Random opponent O
        opponent_action = random.choice([
            i for i in range(9) if env.board[i] == ' '
        ])

        env.make_move(opponent_action, 'O')

        if env.check_winner('O'):
            losses += 1
            break

        if ' ' not in env.board:
            draws += 1
            break

# Display results
print("\nTesting Results")
print("Wins:", wins)
print("Losses:", losses)
print("Draws:", draws)

print("Win Percentage:", wins / test_games * 100, "%")

# Display one sample game board
print("\nSample Final Board:")
env.display()